<a href="https://colab.research.google.com/github/FarhatAsharfillah/weather-prediction-xgboost-randomforest/blob/main/XGBoost_RandomForest_Weather_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Upload file kaggle.json milikmu
from google.colab import files
files.upload()

# 2. Membuat folder rahasia untuk API Key Kaggle agar terbaca oleh sistem
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Mendownload dataset
!kaggle datasets download -d jsphyg/weather-dataset-rattle-package

# 4. Mengekstrak file zip
!unzip weather-dataset-rattle-package.zip

Saving weatherAUS.csv to weatherAUS.csv
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package
License(s): other
100% 3.83M/3.83M [00:00<00:00, 201MB/s]

Archive:  weather-dataset-rattle-package.zip
replace weatherAUS.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: weatherAUS.csv          


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, mean_absolute_error
from imblearn.over_sampling import SMOTE

# ==========================================
# FASE 1 & 2: LOAD DATA & PRA-PEMROSESAN
# ==========================================
# Membaca file CSV asli dari Kaggle
df = pd.read_csv('weatherAUS.csv')

# Menghapus baris yang datanya kosong (Data Cleaning)
df = df.dropna()

# Memilih kolom asli dari dataset WeatherAUS sebagai X dan y
X = df[['MinTemp', 'MaxTemp', 'Humidity3pm', 'Pressure3pm', 'WindSpeed3pm']]
y = df['RainTomorrow'] # Target: Apakah besok hujan? (Yes/No)

# Mengubah target teks 'Yes'/'No' jadi angka 1/0
y = y.map({'Yes': 1, 'No': 0})

print("Distribusi Kelas Sebelum SMOTE:")
print(y.value_counts())

# Membagi Data Latih (80%) dan Data Uji (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# SMOTE (Hanya diaplikasikan pada Data Latih!)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nDistribusi Kelas Data Latih SETELAH SMOTE:")
print(y_train_smote.value_counts())

# ==========================================
# FASE 3: PEMODELAN
# ==========================================
# Inisialisasi Model
rf_model = RandomForestClassifier(random_state=42)
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Melatih Model menggunakan data yang sudah di-SMOTE
print("\nMelatih model Random Forest...")
rf_model.fit(X_train_smote, y_train_smote)

print("Melatih model XGBoost...")
xgb_model.fit(X_train_smote, y_train_smote)

# Meminta model menebak Data Uji (Testing Data yang murni)
rf_predictions = rf_model.predict(X_test)
xgb_predictions = xgb_model.predict(X_test)

# ==========================================
# FASE 4: EVALUASI KINERJA
# ==========================================
def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    print(f"--- Hasil Evaluasi: {name} ---")
    print(f"Akurasi   : {acc * 100:.2f}%")
    print(f"F1-Score  : {f1:.4f}")
    print(f"RMSE      : {rmse:.4f}")
    print(f"MAE       : {mae:.4f}\n")

# Cetak hasil komparasi
print("\n" + "="*40)
evaluate_model("Random Forest", y_test, rf_predictions)
evaluate_model("XGBoost", y_test, xgb_predictions)
print("="*40)

Distribusi Kelas Sebelum SMOTE:
RainTomorrow
0    43993
1    12427
Name: count, dtype: int64

Distribusi Kelas Data Latih SETELAH SMOTE:
RainTomorrow
1    35194
0    35194
Name: count, dtype: int64

Melatih model Random Forest...
Melatih model XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:37:12] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Hasil Evaluasi: Random Forest ---
Akurasi   : 81.73%
F1-Score  : 0.5868
RMSE      : 0.4275
MAE       : 0.1827

--- Hasil Evaluasi: XGBoost ---
Akurasi   : 82.10%
F1-Score  : 0.5998
RMSE      : 0.4231
MAE       : 0.1790

